In [1]:
import sys
sys.path.insert(0, "/users/eleves-a/2025/iuliia.korotkova/VLM_probing")

from PIL import Image
from src.probing.probe import load_probe, load_best_probe, predict_with_probe
from src.extraction.extract import extract_single, MODEL_REGISTRY

In [2]:
import os
os.environ["HF_HOME"] = "/Data/iuliia.korotkova/huggingface_cache"

In [3]:
!echo $HF_HOME

/Data/iuliia.korotkova/huggingface_cache


In [4]:
probes_dir = "/Data/iuliia.korotkova/VLM_data/results/qwen2_spatial/probes"
layer = None

# Load probe
if layer is not None:
    probe, le = load_probe(probes_dir, layer)
    layer_idx = layer
    print(f"Using probe from layer {layer_idx}")
else:
    probe, le, layer_idx = load_best_probe(probes_dir)

Loaded best probe: layer 24 (acc=0.9250)


In [5]:
model_tag = "qwen2"
model_id = None

# Load model
registry_entry = MODEL_REGISTRY[model_tag]
model_id = model_id or registry_entry["default_id"]
print(f"Loading model: {model_id}")
model, processor = registry_entry["loader"](model_id)

Loading model: Qwen/Qwen2-VL-7B-Instruct


/users/eleves-a/2025/iuliia.korotkova/VLM_probing/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.43it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [8]:
image_path = "/users/eleves-a/2025/iuliia.korotkova/VLM_probing/data/raw/color/images/color_00147.png"
prompt = "The color of the the star in the image is"

# Extract hidden states
image = Image.open(image_path).convert("RGB")
repr_array = extract_single(model, processor, model_tag, image, prompt)
# repr_array: (n_layers, hidden_dim)

# Get prediction from the chosen layer
hidden_state = repr_array[layer_idx]
result = predict_with_probe(probe, le, hidden_state)

print(f"\nPrediction: {result['prediction']}")
print(f"Probabilities:")
for cls, prob in sorted(result["probabilities"].items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 30)
    print(f"  {cls:12s} {prob:.4f} {bar}")


Prediction: yellow
Probabilities:
  yellow       0.9977 █████████████████████████████
  green        0.0011 
  blue         0.0010 
  pink         0.0002 
  orange       0.0000 
  brown        0.0000 
  red          0.0000 
  gray         0.0000 
  cyan         0.0000 
  purple       0.0000 
